In [2]:
import json
import pandas as pd
import re

from bs4 import BeautifulSoup

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

## Datakvalitetskontroll inför städning

In [3]:
print("Antal annonser:", len(df))
print("Unika annons-ID:", df["id"].nunique())

Antal annonser: 6889
Unika annons-ID: 6632


In [4]:
# Identifierar annonser som saknar description eller endast innehåller tomrum
missing_description = df[
    df["description"].isna() |
    df["description"].fillna("").str.strip().eq("")
]

print("Antal annonser utan description:", len(missing_description))

missing_description[
    ["id", "title", "employer", "occupation", "description"]
].head(10)

Antal annonser utan description: 8


,id,title,employer,occupation,description
931,29506534,Teamcenter Solution Architect operations,Hitachi Energy Sweden AB,IT-arkitekt/Lösningsarkitekt,
2424,29737028,Core Network Analyst,Semicon Service Nordic AB,Systemanalytiker/Systemutredare,
2824,29817808,Senior Power BI Specialist - Älmhult,Danda AB,"Affärskonsult, IT",
4475,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,IT-utredare,
4903,30600803,Dynamics 365 Architect / D365 Solution Archite...,Veritaz AB,Enterprise Architect,
5455,30854632,Power BI Consultant (Report Development),Semicon Service Nordic AB,"Verksamhetskonsult, IT",
5776,30961171,Senior Data Engineer,Semicon Service Nordic AB,Data scientist,
5824,30983013,Linux Engineer,Ericsson AB,Systemanalytiker/Systemutredare,


In [5]:
# Identifierar alla rader som tillhör ett duplicerat annons-ID
duplicate_ids = df[
    df["id"].duplicated(keep=False)
].sort_values("id")

print("Antal rader som tillhör ett duplicerat ID:", len(duplicate_ids))
print("Antal duplicerade ID:", duplicate_ids["id"].nunique())

duplicate_ids[
    ["id", "version_id", "title", "employer", "published"]
].head(10)


Antal rader som tillhör ett duplicerat ID: 475
Antal duplicerade ID: 218


,id,version_id,title,employer,published
4,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,2025-01-02T10:15:14
5,29354585,f0883b87209ad585259381a396b05e5618ea3d93,Projektledare,Randstad AB,2025-01-02T10:15:14
30,29361752,27b784e84eb1a5703befb304f9f76c3a3c8797db,Teknikansvarig VA till SYSTRA AB i Jönköping,SYSTRA AB,2025-01-07T13:27:42
31,29361752,5fddf1db951f3374f7792910c4f425650ce236e6,Teknikansvarig VA till SYSTRA i Jönköping,SYSTRA AB,2025-01-07T13:27:42
37,29362368,98e4dc9c1ec0419c0fcef0319b64909342df680d,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
38,29362368,aa58e8f5ebbe56dd9dbc558eea70eb14085eb6de,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
39,29362443,4acb1a3da382edaf553e92b57524ad381d704d85,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
40,29362443,a335ce8a599380486cb101644acedcc27bef5d5d,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
50,29363892,de9e985fb72fb75014b4c4f676ce7c8bee93db9e,Senior Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04
51,29363892,e3e83d3e1186b260000ba39f70d40dbb5265842d,Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04


In [6]:
# Identifierar HTML-taggar som kan utgöra brus i NLP-analysen
html_pattern = r"<[^>]+>"

html_ads = df[
    df["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal annonser med möjlig HTML:", len(html_ads))

html_ads[
    ["id", "title", "description"]
].head(10)

Antal annonser med möjlig HTML: 3


,id,title,description
4717,30542907,Infrastrukturarkitekt till Växjö kommun,Vill du arbeta med teknik som gör verklig skil...
4966,30622455,Chefsarkitekt | Jula | Skara,Är du redo att forma Julas digitala framtid? D...
5606,30911356,Lösningsarkitekt,Vill du vara med och forma framtidens digitala...


In [7]:
# Regex för att identifiera personnummer
personnummer_pattern = r"\b(?:19|20)?\d{6}[-+ ]?\d{4}\b"

personnummer_count = df["description"].fillna("").str.contains(
    personnummer_pattern, regex=True
).sum()

print("Annonser med möjliga personnummer:", personnummer_count)

Annonser med möjliga personnummer: 0


In [8]:
# Regex för att identifiera epost-adresser
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

email_ads = df[
    df["description"]
    .fillna("")
    .str.contains(email_pattern, regex=True)
]

print("Antal annonser med e-post:", len(email_ads))


Antal annonser med e-post: 0


In [9]:
# Regex för att identifiera möjliga svenska telefonnummer
phone_pattern = r"(?<!\d)(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,11}(?!\d)"

phone_ads = df[
    df["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print("Antal annonser med möjliga telefonnummer:", len(phone_ads))


Antal annonser med möjliga telefonnummer: 24


In [10]:
# Kontroll av möjliga telefonnummer före rensning
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

phone_results = df.loc[
    phone_matches.str.len() > 0,
    ["id", "title", "employer"]
].copy()

phone_results["phone_match"] = phone_matches[
    phone_matches.str.len() > 0
].apply(lambda x: ", ".join(x))

phone_results

,id,title,employer,phone_match
186,29386902,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
187,29386904,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
188,29386905,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
427,29422415,Senior Kravanalytiker,Deploja AB,03-02-2025\n
535,29439711,"IT-arkitekt, inriktning Digital Arbetsplats",Länsstyrelsen i Västra Götalands län,000-59 000
1045,29533240,Dataanalytiker sökes av Göteborgs Spårvägar,Göteborgs Stads Kollektivtrafik AB,000-52 000
1865,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
1866,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
2667,29791470,Senior IT-arkitekt,Deploja AB,01-07-2025
3048,29929162,IT Systemförvaltare – Teknikstöd nät och platt...,Försvarsmakten,0589-40400


In [11]:
# Identifierar möjliga telefonnummer i annonstexterna med hjälp av regex
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

# Hämtar de annonser där minst ett möjligt telefonnummer har hittats
phone_ads = df.loc[
    phone_matches.str.len() > 0
].copy()

# Visar telefonträffarna tillsammans med texten runt träffen för manuell granskning
for _, row in phone_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Träff:", phone_matches.loc[row.name])
    
    description = row["description"]
    
    for match in phone_matches.loc[row.name]:
        start = description.find(match)
        context = description[max(0, start - 100): start + len(match) + 100]
        print("Sammanhang:")
        print(context)

ID: 29386902
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386904
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386905
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29422415
Titel: Senior Kravanalytiker
Träff: ['03-02-2025\n']
Sammanhang:
n av

## Cleaning

In [12]:
print("Antal annonser före rensning:", len(df))
print("Antal unika ID före rensning:", df["id"].nunique())


# Behåller endast en annons per unikt annons-ID
df_clean = df.drop_duplicates(subset="id", keep="first").copy()


print("-------------------------------------------")
print("Antal annonser efter rensning:", len(df_clean))
print("Antal unika ID efter rensning:", df_clean["id"].nunique())

Antal annonser före rensning: 6889
Antal unika ID före rensning: 6632
-------------------------------------------
Antal annonser efter rensning: 6632
Antal unika ID efter rensning: 6632


In [13]:
# Identifierar annonser som saknar annonstext
missing_description = df_clean[
    df_clean["description"].isna() |
    df_clean["description"].fillna("").str.strip().eq("")
]

print("Annonser utan description efter rensning:", len(missing_description))

Annonser utan description efter rensning: 8


In [14]:
#  Tar bort annonser som saknar annonstext
df_clean = df_clean[
    df_clean["description"].fillna("").str.strip().ne("")
].copy()

print("Antal annonser efter borttagning:", len(df_clean))

Antal annonser efter borttagning: 6624


In [15]:
# Kontrollerar om möjliga telefonnummer fortfarande finns kvar efter städning
phone_ads_after_duplicates = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print(
    "Annonser med möjlig telefonträff efter dublettrensning:",
    len(phone_ads_after_duplicates)
)

Annonser med möjlig telefonträff efter dublettrensning: 23


In [16]:
# Ersätter verifierade telefonnummer med neutral markör 
def clean_phone_numbers(text):
    if pd.isna(text):
        return text
    
    return re.sub(phone_pattern, "[PHONE]", text)


df_clean["description"] = df_clean["description"].apply(clean_phone_numbers)


phone_matches_clean = df_clean["description"].fillna("").str.findall(phone_pattern)

print(
    "Antal möjliga telefonträffar efter städning:",
    phone_matches_clean.str.len().sum()
)

Antal möjliga telefonträffar efter städning: 0


In [17]:
# Räknar hur många telefonnummer som ersattes med [PHONE]
phone_replacements = df_clean["description"].fillna("").str.count(r"\[PHONE\]").sum()

print("Antal ersatta telefonnummer:", phone_replacements)

Antal ersatta telefonnummer: 23


In [18]:
# Kontrollerar om några HTML-taggar fortfarande finns kvar efter cleaning
html_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal HTML-annonser efter städning:", len(html_ads))

Antal HTML-annonser efter städning: 3


In [19]:
# Visar annonser med kvarvarande HTML för manuell kontroll
for _, row in html_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print(row["description"])

ID: 30542907
Titel: Infrastrukturarkitekt till Växjö kommun
Vill du arbeta med teknik som gör verklig skillnad? Hos oss får du chansen att påverka utvecklingen av en hel kommun, där smarta digitala lösningar bidrar till en enklare, tryggare och mer hållbar vardag för invånare, företag och besökare. Nu söker vi en infrastrukturarkitekt som vill vara en nyckelspelare i vår tekniska utvecklingsresa och som motiveras av att arbeta i ett komplext och samhällsviktigt IT‑landskap. Växjö kommun är Kronobergs största arbetsgivare och varje dag arbetar vi tillsammans för framtidens samhälle. <img src="https://sae3ndp007.blob.core.windows.net/swe/71d22a21-1a8c-479b-8fac-9af5f5be811b.jpg">
 
 
Om rollen 
Som infrastrukturarkitekt blir du en del av vårt team Arkitektur & Utveckling, tillsammans med 12 kollegor inom infrastruktur-, nät- och lösningsarkitektur samt systemutveckling. Här får du en central roll i att forma vår tekniska riktning och skapa robusta och säkra lösningar. Du får arbeta i en 

In [20]:
# Tar bort HTML-taggar men behåller den synliga texten
def clean_html(text):
    if pd.isna(text):
        return text
    
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)


df_clean["description"] = df_clean["description"].apply(clean_html)


html_after = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
]

print("Annonser med HTML efter cleaning:", len(html_after))

Annonser med HTML efter cleaning: 0


In [21]:
# Söker efter kontaktrelaterade fraser för att manuellt kontrollera
# om personnamn eller andra personuppgifter förekommer i anslutning till dessa
contact_pattern = (
    r"(?i)(kontaktperson|kontakt|kontakta|ansvarig|rekryterare|"
    r"rekryteringsansvarig|för frågor|vid frågor|har du frågor)"
)

contact_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(contact_pattern, regex=True)
]

print("Annonser med kontaktfraser:", len(contact_ads))

for _, row in contact_ads.head(20).iterrows():
    description = row["description"]
    match = re.search(contact_pattern, description)

    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])

    if match:
        start = match.start()
        end = min(len(description), match.end() + 200)

        print(description[start:end].replace("\n", " "))

C:\Users\Belson\AppData\Local\Temp\ipykernel_15104\2175045008.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(contact_pattern, regex=True)


Annonser med kontaktfraser: 1744
ID: 29354585
Titel: Projektledare
kontaktytor och intressanta utmaningar. Då kan denna tjänst vara något för dig. Beskrivning av gruppen, En hel kommun är beroende av att vi genomför vårt uppdrag i allt från naturnära områden till stadsmiljö
ID: 29355065
Titel: Consultant Manager till ett av våra grymma Dynamics 365 F&SCM-team
kontakt och involveras tidigt i affärerna ur ett resurssättningsperspektiv. Viktigt att poängtera är att vi inte jobbar med resurskonsulter, utan att vi är ett projektkonsultföretag vilket gör att dina medar
ID: 29355172
Titel: Systemspecialist med inriktning IT-infrastruktur till Säkra IT-lösningar
kontakter. Önskar du att arbeta delvis på distans? I den här tjänsten kan det finnas möjlighet att arbeta hemifrån upp till 40 procent av din arbetstid. #LI-Hybrid Intervjuer sker via Skype, preliminärt unde
ID: 29355229
Titel: IT arkitekt / Lösningsarkitekt till energisektorn i Umeå
ansvarig för Lösningsarkitekturteamet. Uppdragsperio

- Vid kontroll av potentiella personuppgifter identifierades inga träffar för e-postadresser eller personnummer.  

- Efter rensning av dubletter identifierades 8 annonser som saknade användbar `description`. Dessa annonser tas bort eftersom de inte innehåller någon text som kan användas i kompetensanalysen. 

- Regexbaserad identifiering av telefonnummer gav 23 möjliga träffar. Dessa granskades manuellt för att bedöma om de utgjorde faktiska telefonnummer.   
Några av träffarna visade sig vara telefonnummer till företagsväxlar. Verifierade telefonnummer, inklusive företagsväxlar, ersätts med `[PHONE]` eftersom de inte är relevanta för kompetensanalysen.

- Förekomst av HTML-taggar kontrollerades med regex. Ett fåtal annonser innehöll HTML-formatering, men mängden var begränsad. HTML-taggarna tas bort eftersom de inte innehåller relevant information för kompetensanalysen och annars kan utgöra brus vid den fortsatta NLP-bearbetningen.

- Som en extra kontroll av personnamn söktes kontaktrelaterade fraser i annonstexterna för att undersöka om man kan hitta personnamn. Vid manuell granskning syntes inga direkta personuppgifter, utan i de fall där sådan information verkar ha funnits var den redan ersatt med `****`.     
Det tyder på att personuppgifter redan har maskerats i materialet som hämtats från JobTech Historical API.

- Regexbaserad identifiering kan användas för strukturerade uppgifter som e-postadresser, telefonnummer och personnummer men är mindre lämplig för att identifiera personnamn. Förekomst av personnamn kommer därför undersökas separat med spaCy NER. 



## Identifiering av personnamn med SpaCy

Personuppgifter återfinns vanligen i slutet av annonser i relation till "för fler frågor, kontakta.." eller "rekryterande chef" efterföljt av ett namn. Därför börjar vi med att ladda in texten och tittar på de 300 sista orden i ett urval av annonserna för att se om det finns några personnamn.

In [26]:
print(len(df_clean))

# slumpa 10 annonser och titta på de sista 300 tecknen
sample = df_clean.dropna(subset=["description"]).sample(10, random_state=1)
for _, row in sample.iterrows():
    print(row["id"], "...", row["description"][-300:])
    print("-" * 60)

6624
30389453 ... Frågorna utgår från kravprofilen för den här rekryteringen och dina svar ligger till grund för det urval som sker i processen. Att besvara frågorna är en förutsättning för att din ansökan ska anses komplett. Vänligen notera att krav och eventuella meriter ska vara uppfyllda vid ansökningstillfället.
------------------------------------------------------------
31063475 ... nsyn till. Omfattning: Heltid, kontorstider – vi är flexibla och utgår från att det ska fungera för alla parter. Placering: Malmö, Helsingborg, Halmstad, Göteborg, Jönköping, Växjö eller Stockholm. Rekryteringsprocess: Inledande onlineintervju/samtal, djupintervjuer, tester/case och referenstagning.
------------------------------------------------------------
29407615 ... h safety concept development and implementation. Ability to work independently and deliver results in a fast-paced project environment. Excellent communication and collaboration skills to work effectively with cross-functional teams

Utskrifter visar att inga personuppgifter dyker upp i någon av exemplerna. Kan bero på att det antingen inte finns några personuppgifter alls i exempelannonserna (redan borttagna av JobTech) eller att de finns någon annanstans än i slutet av annonstexten. 

### SpaCy pipeline

Skapa en spaCy pipeline där modellen tittar på ett litet urval och på alla entitetstyper, inte bara personer för att identifiera personnamn.

Vad vi letar efter: 

- Riktiga namn
- Falska träffar
- Missade namn
- Positionerna

In [27]:
import spacy
from collections import Counter

nlp = spacy.load("sv_core_news_sm", disable=["parser", "lemmatizer"])
print(nlp.get_pipe("ner").labels)       # vilka etiketter finns

sample50 = df_clean.sample(50, random_state=1)

counts = Counter()
per_hits = []                           # (annons-id, träff, var i texten 0-1, omgivande text)

for _, row in sample50.iterrows():
    text = row["description"]
    doc = nlp(text)
    for ent in doc.ents:
        counts[ent.label_] += 1
        if ent.label_ == "PRS":     
            pos = ent.start_char / len(text)
            context = text[max(ent.start_char - 40, 0): ent.end_char + 40].replace("\n", " ")
            per_hits.append((row["id"], ent.text, round(pos, 2), context))

print(counts)
print(len(per_hits), "personträffar")
print(Counter(h[1] for h in per_hits).most_common(30))
for h in per_hits[:30]:
    print(h)

('EVN', 'LOC', 'MSR', 'OBJ', 'ORG', 'PRS', 'TME', 'WRK')
Counter({'PRS': 290, 'ORG': 150, 'LOC': 129, 'WRK': 14, 'OBJ': 9, 'TME': 8, 'MSR': 3, 'EVN': 1})
290 personträffar
[('solutions', 25), ('with', 23), ('Experience', 8), ('careers', 7), ('With', 7), ('Europe', 7), ('Collaborate with', 6), ('Kubernetes', 6), ('Sopra Steria', 6), ('Experience with', 5), ('Fellowmind', 3), ('Enterprise', 3), ('Leda', 3), ('workshops', 3), ('Familiarity with', 3), ('Java', 3), ('Avaron AB', 3), ('This', 3), ('Wellness', 3), ('Randstad Digital', 3), ('Ansök', 2), ('Intune', 2), ('Computer Science', 2), ('Knowledge', 2), ('Together', 2), ('Kafka', 2), ('Förmåga', 2), ('Kanban', 2), ('Great Place', 2), ('Winshuttle Studio', 2)]
('30389453', 'Ansök', 0.8, '-01-31 Sista ansökningsdag : 2026-01-08 Ansök genom att bifoga ditt CV (vi vill inte ')
('31063475', 'Fellowmind', 0.01, ' att läsa och sök idag! Om tjänsten Hos Fellowmind jobbar vi sällan i rena ERP‑projekt. I ')
('31063475', 'Enterprise', 0.21, 'm”. D

### Vad resultatet visar

Av de träffar som syns är ingen ett personnamn, vilket tyder på att det flesta träffar faktiskt är falsk positiv. De vanligaste är solutions, with, workshops, careers, Europe, företagsnamn som Avaron AB, och teknikord som Azure, Python, Kubernetes och Visual Studio. 

Noll riktiga namn bland urvalen kan betyda två olika saker:

1. Namnen finns inte i texten, eftersom JobTech tagit bort dem.
2. Namnen finns, men modellen är för dålig på jobbannonser för att hitta dem.

### Litet test

Testar om modellen över huvud taget kan hitta namn i en annons med hjälp av exempeltext. 

In [28]:
test_names = [
    "Kontakta rekryterande chef Anna Svensson på telefon 070 123 45 67.",
    "Har du frågor om tjänsten? Hör av dig till Erik Lindqvist.",
    "Fackliga företrädare: Maria Johansson och Karl Andersson.",
    "Vid frågor kontakta Fatima Hassan, rekryterare.",
]

base = df_clean["description"].iloc[0]
for sentence in test_names:
    doc = nlp(base[-300:] + " " + sentence)
    print([(e.text, e.label_) for e in doc.ents if e.start_char >= len(base[-300:])])

[('Anna Svensson', 'PRS')]
[('Erik Lindqvist', 'PRS')]
[('Maria Johansson', 'PRS'), ('Karl Andersson', 'PRS')]
[('Fatima Hassan', 'PRS')]


Modellen hittar testnamnen vilket tyder på att modellen bör kunna identifiera personnamn i annonstexterna, även om flest träffar verkar vara falska.

Testar att köra pipelinen på alla annonser för att säkerställa att inga personnamn har missats att rensas bort. 

In [29]:
nlp = spacy.load("sv_core_news_sm", disable=["parser", "lemmatizer"])
print(nlp.get_pipe("ner").labels)       # vilka etiketter finns

counts = Counter()
per_hits = []                           # (annons-id, träff, var i texten 0-1, omgivande text)

for _, row in df_clean.iterrows():
    text = row["description"]
    doc = nlp(text)
    for ent in doc.ents:
        counts[ent.label_] += 1
        if ent.label_ == "PRS":     
            pos = ent.start_char / len(text)
            context = text[max(ent.start_char - 40, 0): ent.end_char + 40].replace("\n", " ")
            per_hits.append((row["id"], ent.text, round(pos, 2), context))

print(counts)
print(len(per_hits), "personträffar")
print(Counter(h[1] for h in per_hits).most_common(30))
for h in per_hits[:30]:
    print(h)

('EVN', 'LOC', 'MSR', 'OBJ', 'ORG', 'PRS', 'TME', 'WRK')
Counter({'PRS': 37014, 'ORG': 22044, 'LOC': 21577, 'WRK': 1874, 'TME': 1320, 'OBJ': 865, 'MSR': 240, 'EVN': 186})
37014 personträffar
[('solutions', 3065), ('with', 2977), ('With', 952), ('This', 722), ('Europe', 617), ('careers', 616), ('Experience', 575), ('workshops', 558), ('Experience with', 526), ('• Erfarenhet', 437), ('Avaron AB', 437), ('Collaborate with', 430), ('Kubernetes', 342), ('Familiarity with', 325), ('Together', 272), ('Azure', 267), ('machine', 238), ('Computer Science', 235), ('Define', 226), ('great', 206), ('Förmåga', 199), ('Sopra Steria', 193), ('Leda', 179), ('Collaborate', 178), ('Enterprise Architect', 177), ('Databricks', 170), ('LinkedIn', 164), ('questions', 156), ('Data Scientist', 155), ('Java', 154)]
('29354136', 'solutions', 0.01, 'Veritaz is a leading IT staffing solutions provider in Sweden, committed to advanc')
('29354136', 'careers', 0.04, 'eden, committed to advancing individual careers an

### Filter för att hitta riktiga personnamn bland alla identifierade personträffar

In [34]:
from collections import Counter

# Gör om träfflistan till en tabell
hits = pd.DataFrame(per_hits, columns=["id", "name", "pos", "context"])

# Filter 1: ser det ut som ett namn?
NAME_LIKE = re.compile(r"^[A-ZÅÄÖ][a-zåäö]+(?: [A-ZÅÄÖ][a-zåäö]+){0,2}$")
hits = hits[hits["name"].apply(lambda n: bool(NAME_LIKE.match(n)))]
print(len(hits), "träffar efter strukturfilter")

# Filter 2: används ordet som ett vanligt ord?
lower_counts = Counter()
for text in df_clean["description"]:
    lower_counts.update(re.findall(r"\b[a-zåäö]{2,}\b", text))

def is_common(name, min_count=20):
    return all(lower_counts[w.lower()] >= min_count for w in name.split())

hits["common_word"] = hits["name"].apply(is_common)

skills = json.load(open("../data/skills_2511.json", encoding="utf-8"))
skill_words = {s["skill"].lower() for s in skills}
hits["is_skill"] = hits["name"].str.lower().isin(skill_words)

# Filter 3: står träffen där en kontaktperson brukar stå?
cue = re.compile(r"kontakt|rekryter|frågor|ansvarig|hör av|ring |mejla|maila|fackliga",
                 re.IGNORECASE)
hits["near_cue"] = hits["context"].apply(lambda c: bool(cue.search(c)))
hits["late"] = hits["pos"] > 0.6

# Nivåer att läsa
tier1 = hits[hits["near_cue"] & hits["late"] & ~hits["is_skill"]]
tier2 = hits[~hits["common_word"] & ~hits["is_skill"] & ~hits.index.isin(tier1.index)]
print(len(tier1), "i nivå 1,", len(tier2), "i nivå 2")

20861 träffar efter strukturfilter
695 i nivå 1, 7975 i nivå 2


In [40]:
review = (tier2.groupby("name")
               .agg(annonser=("id", "nunique"), exempel=("context", "first"))
               .sort_values("annonser", ascending=True))

print(len(review), "unika strängar")
print(review.head(200).to_string())

1643 unika strängar
                                     annonser                                                                                                              exempel
name                                                                                                                                                              
Adobe Sign                                  1                           and e-Signing platforms (e.g. DocuSign, Adobe Sign, IBM FileNet, SharePoint, or equivalent
Adobe Tags                                  1                           ng tools such as Google Tag Manager and Adobe Tags (Launch). OR Experience in JavaScript (
Privileged Access Security                  1           om ett eller flera av följande områden: Privileged Access Security Access Management och PAM Identity Gove
Privileged Session Manager                  1           vilegierade konton Implementera en PSM (Privileged Session Manager) för övervakning och kontroll av privil
Ab